In [0]:
TABLE_PRODUCT_BRONZE = "customer_360.bronze.products"
TABLE_SILVER_PRODUCT = "customer_360.silver.products"
TABLE_QUARANTINE_PRODUCT = "customer_360.quarantine.products"
TABLE_PRODUCT_VIEWS = "customer_360.bronze.product_views"
PRODUCT_METRICS_TABLE = "customer_360.raw.product_silver_metrics"
PATH_PRODUCT_CHECKPOINTLOCATION_SILVER = (
    "/Volumes/customer_360/raw/source_files/checkpoints/silver/products"
)
TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS customer_360.silver
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS customer_360.silver.products (

    product_id STRING NOT NULL,
    product_name STRING,
    product_category STRING,
    product_subcategory STRING,
    product_price DECIMAL(12,2),
    product_status STRING,
    updated_at TIMESTAMP

)
USING DELTA
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS customer_360.quarantine
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS customer_360.quarantine.products (

    product_id STRING,
    product_name STRING,
    product_category STRING,
    product_subcategory STRING,
    product_price DECIMAL(12,2),
    product_status STRING,
    updated_at TIMESTAMP,
    failure_reason STRING,
    quarantined_at TIMESTAMP

)
USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS customer_360.raw.product_silver_metrics (

    metric_time TIMESTAMP,
    batch_id BIGINT,
    query_name STRING,
    total_records BIGINT,
    valid_records BIGINT,
    invalid_records BIGINT,
    duplicate_records BIGINT

)
USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS customer_360.bronze.product_views (
    product_id STRING NOT NULL,
    updated_at TIMESTAMP,
    viewed_at TIMESTAMP
)
USING DELTA
""")

In [0]:
product_df = (
    spark
    .readStream
    .format("delta")
    .table(TABLE_PRODUCT_BRONZE)
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime


def process_product_dataframe(batch_df, batch_id):

    df = (
        batch_df

        .withColumn(
            "product_id",
            trim(col("product_id"))
        )

        .withColumn(
            "product_name",
            trim(col("product_name"))
        )

        .withColumn(
            "product_category",
            trim(col("product_category"))
        )

        .withColumn(
            "product_subcategory",
            trim(col("product_subcategory"))
        )

        .withColumn(
            "product_status",
            trim(col("product_status"))
        )

        .withColumn(
            "failure_reason",

            when(
                col("product_id").isNull(),
                "product_id is null"
            )

            .when(
                col("product_name").isNull(),
                "product_name is null"
            )

            .when(
                col("product_category").isNull(),
                "product_category is null"
            )

            .when(
                col("product_subcategory").isNull(),
                "product_subcategory is null"
            )

            .when(
                col("product_price").isNull(),
                "product_price is null"
            )

            .when(
                col("product_price") <= 0,
                "product_price must be greater than 0"
            )

            .when(
                col("product_status").isNull(),
                "product_status is null"
            )

            .when(
                ~col("product_status").isin(
                    "Active",
                    "Inactive",
                    "Discontinued"
                ),
                "invalid product_status"
            )

            .when(
                col("updated_at").isNull(),
                "updated_at is null"
            )

            .otherwise(None)
        )
    )

    
    # Invalid records
    

    valid_data = (
        df
        .filter(col("failure_reason").isNull())
        .drop("failure_reason")
    )

    quarantine_data = (
        df
        .filter(col("failure_reason").isNotNull())
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    quarantine_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_QUARANTINE_PRODUCT)

    
    # Within-batch duplicates
    

    window = (
        Window
        .partitionBy(
            ["product_id", "updated_at"]
        )
        .orderBy(
            col("updated_at").asc()
        )
    )

    valid_data = (
        valid_data
        .withColumn(
            "rn",
            row_number().over(window)
        )
    )

    unique_data = (
        valid_data
        .filter(col("rn") == 1)
        .drop("rn")
    )

    duplicate_data = (
        valid_data
        .filter(col("rn") > 1)
        .drop("rn")
    )

    
    # Previously processed versions
    

    first_occurrence = (
        spark.read
        .format("delta")
        .table(TABLE_PRODUCT_VIEWS)
    )

    seen_data = (
        first_occurrence
        .join(
            unique_data,
            on=["product_id", "updated_at"],
            how="inner"
        )
        .select([
            "product_id",
            "product_name",
            "product_category",
            "product_subcategory",
            "product_price",
            "product_status",
            "updated_at"
        ])
    )

    
    # New Product versions
    

    silver_df = (
        unique_data
        .join(
            first_occurrence,
            on=["product_id", "updated_at"],
            how="left_anti"
        )
        .select([
            "product_id",
            "product_name",
            "product_category",
            "product_subcategory",
            "product_price",
            "product_status",
            "updated_at"
        ])
    )

    
    # Write Silver
    

    silver_df.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_SILVER_PRODUCT)

    valid_data_count = silver_df.count()

    
    # Duplicate quarantine
    

    quarantine_data = (
        duplicate_data
        .unionByName(seen_data)
        .withColumn(
            "failure_reason",
            lit("duplicate record")
        )
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    quarantine_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_QUARANTINE_PRODUCT)

    duplicate_records = quarantine_data.count()

    
    # Track processed versions
    

    viewed_data = (
        silver_df
        .withColumn(
            "viewed_at",
            current_timestamp()
        )
        .select([
            "product_id",
            "updated_at",
            "viewed_at"
        ])
    )

    viewed_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_PRODUCT_VIEWS)

    
    # Silver metrics
    

    metric = [
        Row(
            metric_time=datetime.now(),
            batch_id=batch_id,
            query_name="product_silver",
            total_records=batch_df.count(),
            valid_records=valid_data_count,
            invalid_records=(
                batch_df.count() - valid_data_count
            ),
            duplicate_records=duplicate_records
        )
    ]

    metric_df = spark.createDataFrame(metric)

    metric_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(PRODUCT_METRICS_TABLE)

In [0]:
query = (
    product_df
    .writeStream
    .trigger(availableNow=True)
    .foreachBatch(process_product_dataframe)
    .option(
        "checkpointLocation",
        PATH_PRODUCT_CHECKPOINTLOCATION_SILVER
    )
    .start()
)

query.awaitTermination()

In [0]:
import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="product_silver",
            batch_id=int(progress["batchId"]),
            input_rows=int(
                source.get("numInputRows", 0)
            ),
            input_rows_per_second=float(
                source.get("inputRowsPerSecond", 0.0)
            ),
            processed_rows_per_second=float(
                source.get("processedRowsPerSecond", 0.0)
            ),
            processing_time_ms=int(
                progress
                .get("durationMs", {})
                .get("triggerExecution", 0)
            )
        )
    )

if metrics:

    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)

In [0]:
spark.sql("""
SELECT *
FROM customer_360.bronze.products
""").show()

spark.sql("""
SELECT *
FROM customer_360.silver.products
""").show()

spark.sql("""
SELECT *
FROM customer_360.bronze.product_views
""").show()

spark.sql("""
SELECT *
FROM customer_360.quarantine.products
""").show()

spark.sql("""
SELECT *
FROM customer_360.raw.product_silver_metrics
""").show()

spark.sql("""
SELECT *
FROM customer_360.raw.stream_metrics
""").show()